# ✅ Auth + DB Setup


## 🔐 **Authenticate to Google Cloud within Colab**

Authenticate to Google Cloud as the IAM user logged into this notebook in order to access your Google Cloud Project.

In [1]:
from google.colab import auth

auth.authenticate_user()

MessageError: ignored

## 💻 **Install Code Dependencies**
It is recommended to use the Connector alongside a library that can create connection pools, such as [SQLAlchemy](https://www.sqlalchemy.org/).
This will allow for connections to remain open and be reused, reducing connection overhead and the number of connections needed

Let's `pip install` the [Cloud SQL Python Connector](https://github.com/GoogleCloudPlatform/cloud-sql-python-connector) as well as [SQLAlchemy](https://www.sqlalchemy.org/), using the below command.

In [ ]:
# install dependencies
import sys
!{sys.executable} -m pip install cloud-sql-python-connector["pymysql"] SQLAlchemy==2.0.7

In [ ]:
import google.auth
import pandas as pd

from google.cloud.sql.connector import Connector
from google.auth.transport.requests import Request
from sqlalchemy import create_engine, Table, Column, Integer, VARCHAR, ForeignKey, String, MetaData, text, select
from sqlalchemy.orm import Session

## 🐬 **Connect to a MySQL Instance**
We are now ready to connect to a MySQL instance using the Cloud SQL Python Connector! 🐍 ⭐ ☁


In [ ]:
# initialize parameters
INSTANCE_CONNECTION_NAME = "inlaid-woods-388716:us-west1:ilkmaar" # i.e demo-project:us-central1:demo-instance
print(f"Your instance connection name is: {INSTANCE_CONNECTION_NAME}")

# IAM database user parameter (IAM user's email before the "@" sign, mysql truncates usernames)
# ex. IAM user with email "demo-user@test.com" would have database username "demo-user"
# grant Cloud SQL Client role to authenticated user
current_user = !gcloud auth list --filter=status:ACTIVE --format="value(account)"

IAM_USER = current_user[0].split("@")[0]
DB_NAME = "gameplay-data"

### ✅ **Connect to Database**
To connect to Cloud SQL using the connector, initialize a `Connector` object and call its `connect` method with the proper input parameters.

In [ ]:
# initialize connector
connector = Connector()

# getconn now using IAM user and requiring no password with IAM Auth enabled
def getconn():
    conn = connector.connect(
      INSTANCE_CONNECTION_NAME,
      "pymysql",
      user=IAM_USER,
      db=DB_NAME,
      enable_iam_auth=True
    )
    return conn

# create connection pool
engine = create_engine(
    "mysql+pymysql://",
    creator=getconn,
)

def query_db(query_str):
    with engine.connect() as conn:
        return pd.read_sql_query(text(query_str), conn)

metadata=MetaData()

### Test Connection

This fails if the user does not have access to that database yet.

To fix:
  - log into the MySQL server on the Google Cloud Shell as root user
  - MySQL > GRANT ALL PRIVILEGES on `gameplay-data`.* to "user"@'%'

In [ ]:
# connect to connection pool
with engine.connect() as db_conn:
    # get current datetime from database
    results = db_conn.execute(text("SELECT NOW()")).fetchone()

    # output time
    print("Current time: ", results[0])

# 🌎 Behaviors and Definitions

## Player and creature helper functions

### **Define Creature functions**

In [ ]:
## Define Creature Class

# Has name, id, health, mood
# Has Collection (query for Creature Collection id)

# Consume (create Consumption event)

### **Define Player functions**


In [ ]:
## Define Player Class

# Has name, id, favorite_species, favorite_island

# forages on favorite island most, then island of favorite species, then random
# if recipes can be made from items in inventory, crafts them
# interacts with creatures, sometimes gifts from inventory

In [ ]:
def get_inventory_id(player_id):
    # Create the SQL query
    query = f"""
    SELECT collections.collection_id
    FROM collections
    JOIN player_collection_access ON collections.collection_id = player_collection_access.collection_id
    WHERE player_collection_access.player_id = '{player_id}' AND collections.collection_type = 'inventory';
    """

    # Execute the query and return the result
    with engine.connect() as conn:
        result = pd.read_sql_query(text(query), conn)

    # Assuming there's only one collection per player of type 'player',
    # the following line will return the collection ID as an integer
    return result['collection_id'].values[0]

In [ ]:
def get_crafting_table_id(player_id):
    # Create the SQL query
    query = f"""
    SELECT collections.collection_id
    FROM collections
    JOIN player_collection_access ON collections.collection_id = player_collection_access.collection_id
    WHERE player_collection_access.player_id = '{player_id}' AND collections.collection_type = 'crafting_table';
    """

    # Execute the query and return the result
    with engine.connect() as conn:
        result = pd.read_sql_query(text(query), conn)

    # Assuming there's only one collection per player of type 'player',
    # the following line will return the collection ID as an integer
    return result['collection_id'].values[0]

## Foraging code

In [ ]:
import random

resources = Table('resources', metadata, autoload_with=engine)
resource_transfers = Table('resource_transfers', metadata, autoload_with=engine)

def determine_resources_to_create(favorite_island, favorite_faction):
    resources = []
    types = ['Light', 'Shadow', 'Growth', 'Stability']

    # Fetch resources from the database
    with engine.connect() as conn:
        resources_data = pd.read_sql_query(text("SELECT * FROM resource_types"), conn)

    # Get the id of the map area collection corresponding to the favorite_island and favorite_species
    query = f"""
    SELECT collections.collection_id FROM collections
    JOIN location_groups ON collections.collection_name = location_groups.location_group_name
    WHERE location_groups.location_group_island_faction IN ('{favorite_island}', '{favorite_faction}')
    AND location_groups.location_group_type = 'collection_area';
    """
    with engine.connect() as conn:
        map_collection_ids = pd.read_sql_query(text(query), conn)

    # Create resources
    num_resources = random.randint(5, 10)  # generate a random number of resources between 5 to 10
    for _ in range(num_resources):
        # Choose a type with higher probability for favorite_island and favorite_species
        type_probabilities = [0.1 if t not in (favorite_island, favorite_faction) else 0.4 for t in types]
        resource_type = random.choices(types, weights=type_probabilities, k=1)[0]

        # Choose a resource of the selected type
        chosen_resource = resources_data[resources_data['resource_type_faction'] == resource_type].sample(1)
        resource_type_id = chosen_resource['resource_type_id'].values[0]

        # Choose a random map collection id
        collection_id = random.choice(map_collection_ids['collection_id'])

        # Store the created resource
        resource = {'resource_type_id': resource_type_id, 'collection_id': collection_id}
        resources.append(resource)

    return resources

def simulate_foraging(player, inventory_id, time):
    # Based on the favorite island and species, determine which resources to create
    # You would need to define the logic for this yourself, as it depends on how you've modeled resources in your database
    # Get the player's favorite island and species
    favorite_island = player['player_favorite_island']
    favorite_species = player['player_favorite_faction']
    resources_to_create = determine_resources_to_create(favorite_island, favorite_species)

    # Create a new resource for each the player found
    for resource in resources_to_create:
        r_type_id = resource['resource_type_id']
        c_id = resource['collection_id']

        # Insert the new resource into the player's inventory
        new_resource = create_resource(r_type_id, c_id, random.randint(5, 10))
        transfer_resource(new_resource, c_id, inventory_id, time)

## Crafting Code

In [ ]:
def get_all_recipes():
    results = query_db("""
    SELECT DISTINCT recipe_id, recipe_name, recipe_category
    FROM recipes;
    """)
    return results

def get_recipe_ingredients():
    results = query_db(f"""
      SELECT
        recipes.recipe_id,
        recipe_name,
        recipe_category,
        resource_types.resource_type_id,
        resource_type,
        resource_type_faction,
        resource_type_rarity,
        resource_types.resource_type_base_health_effect,
        resource_types.resource_type_base_mood_effect
      FROM
        recipe_ingredient_resource_types
      JOIN
        recipes ON recipes.recipe_id = recipe_ingredient_resource_types.recipe_id
      JOIN
        resource_types on recipe_ingredient_resource_types.resource_type_id = resource_types.resource_type_id
      """)
    return results;

def get_resources_in_inventory(inventory_id):
    results = query_db(f"""
    SELECT resource_id, resource_type_id, resource_quality
    FROM resources
    WHERE collection_id = {inventory_id};
    """)
    return results

def get_items_in_inventory(inventory_id):
    results = query_db(f"""
    SELECT item_id, recipe_name, recipes.recipe_id, item_quality, recipes.recipe_base_health_effect, recipes.recipe_base_mood_effect
    FROM items
    JOIN recipes on items.recipe_id = recipes.recipe_id
    WHERE collection_id = {inventory_id};
    """)
    return results

In [ ]:
def create_resource(resource_type_id, resource_quality, collection_id):
    resources = Table('resources', metadata, autoload_with=engine)
    with engine.begin() as conn:
        ins = resources.insert().values(resource_type_id=resource_type_id, collection_id=collection_id, resource_quality=resource_quality)
        result = conn.execute(ins)
    return result.inserted_primary_key[0]

def create_item(recipe_id, item_quality, collection_id):
    items = Table('items', metadata, autoload_with=engine)
    with engine.begin() as conn:
        ins = items.insert().values(recipe_id=recipe_id, collection_id=collection_id, item_quality=item_quality)
        result = conn.execute(ins)
    return result.inserted_primary_key[0]

def update_resource_collection(resource_id, new_collection_id):
    query = f"""
        UPDATE resources
        SET collection_id = '{new_collection_id}'
        WHERE resource_id = '{resource_id}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))

def update_item_collection(item_id, new_collection_id):
    query = f"""
        UPDATE items
        SET collection_id = '{new_collection_id}'
        WHERE item_id = '{item_id}';
    """
    with engine.begin() as conn:
        conn.execute(text(query))

def update_creature_data(creature_id, creature_mood, creature_health):
    query = f"""
        UPDATE creatures
        SET creature_mood = :creature_mood,
            creature_health = :creature_health
        WHERE creature_id = :creature_id;
    """
    with engine.begin() as conn:
        conn.execute(text(query), {"creature_id": creature_id, "creature_mood": creature_mood, "creature_health": creature_health})

def update_player_creature_relationship_data(player_id, creature_id, relationship_level):
    query = f"""
        UPDATE player_creature_relationships
        SET player_creature_relationship_level = :relationship_level,
        WHERE creature_id = :creature_id
        AND player_id = :player_id;
    """
    with engine.begin() as conn:
        conn.execute(text(query), {"player_id": player_id, "creature_mood": creature_id, "creature_id": relationship_level})

def boost_relationship_level(player_id, creature_id, boost):
    query = f"""
        UPDATE player_creature_relationships
        SET player_creature_relationship_level = player_creature_relationship_level + :boost
        WHERE creature_id = :creature_id
        AND player_id = :player_id;
    """
    with engine.begin() as conn:
        conn.execute(text(query), {"player_id": player_id, "creature_id": creature_id, "boost": boost})

def create_resource_transfer(item_id, source_id, destination_id, time):
    resource_transfers = Table('resource_transfers', metadata, autoload_with=engine)
    with engine.begin() as conn:
        ins = resource_transfers.insert().values(resource_id=item_id, source_collection_id=source_id, destination_collection_id=destination_id, resource_transfer_time=time)
        result = conn.execute(ins)
    return result.inserted_primary_key[0]

def create_item_transfer(item_id, source_id, destination_id, time):
    item_transfers = Table('item_transfers', metadata, autoload_with=engine)
    with engine.begin() as conn:
        ins = item_transfers.insert().values(item_id=item_id, source_collection_id=source_id, destination_collection_id=destination_id, item_transfer_time=time)
        result = conn.execute(ins)
    return result.inserted_primary_key[0]

def create_crafting_event(crafting_event_skill_level, recipe_id, item_transfer_id, crafting_table_collection_id, crafting_event_time):
    crafting_events = Table('crafting_events', metadata, autoload_with=engine)

    with engine.begin() as conn:
        ins = crafting_events.insert().values(crafting_event_skill_level=crafting_event_skill_level, recipe_id=recipe_id, item_transfer_id=item_transfer_id, crafting_table_collection_id=crafting_table_collection_id, crafting_event_time=crafting_event_time)
        result = conn.execute(ins)
    return result.inserted_primary_key[0]

def create_gifting_event(player, creature, item_transfer_id, time):
    gifting_events = Table('gifting_events', metadata, autoload_with=engine)

    with engine.begin() as conn:
        ins = gifting_events.insert().values(player_id=player['player_id'], creature_id=creature['creature_id'], item_transfer_id=item_transfer_id, gifting_event_time=time)
        result = conn.execute(ins)
    return result.inserted_primary_key[0]

def create_interaction_event(player, creature, location_id, gifting_event_id, time):
    if gifting_event_id:
        query = f"""
            INSERT INTO interaction_events (player_id, creature_id, interaction_event_observed_creature_mood, location_id, gifting_event_id, interaction_event_time)
            VALUES ('{player['player_id']}', '{creature['creature_id']}', '{creature['creature_mood']}', '{location_id}', '{gifting_event_id}', '{time}');
        """
    else:
        query = f"""
            INSERT INTO interaction_events (player_id, creature_id, interaction_event_observed_creature_mood, location_id, interaction_event_time)
            VALUES ('{player['player_id']}', '{creature['creature_id']}', '{creature['creature_mood']}', '{location_id}', '{time}');
        """
    with engine.begin() as conn:
        conn.execute(text(query))

def transfer_resource(item_id, source_id, destination_id, time):
    update_resource_collection(item_id, destination_id)
    resource_transfer_id = create_resource_transfer(item_id, source_id, destination_id, time)
    return resource_transfer_id

def transfer_item(item_id, source_id, destination_id, time):
    update_item_collection(item_id, destination_id)
    item_transfer_id = create_item_transfer(item_id, source_id, destination_id, time)
    return item_transfer_id

In [ ]:
# find all possible recipes to craft
def get_craftable_recipes(inventory_items):
    craftable_recipes = []
    all_recipes = get_all_recipes()
    recipe_ingredients = get_recipe_ingredients()

    inventory_resource_type_ids = set(inventory_items['resource_type_id'].tolist())

    for index, row in all_recipes.iterrows():
        recipe_id = row['recipe_id']
        needed_ingredient_ids = set(recipe_ingredients[recipe_ingredients.recipe_id == recipe_id]['resource_type_id'].tolist())

        # If all ingredients of a recipe are in inventory, append the recipe to craftable recipes
        if needed_ingredient_ids.issubset(inventory_resource_type_ids):
            craftable_recipes.append({'recipe_id': recipe_id, 'recipe_name': row['recipe_name'], 'recipe_category': row['recipe_category'], 'needed_ingredients':needed_ingredient_ids})

    return pd.DataFrame(craftable_recipes)

def craft_items(recipes, available_items):
    crafted_items = []
    for _, recipe in recipes.iterrows():
        # Get the required ingredients for the recipe
        required_resource_type_ids = recipe.needed_ingredients

        # Available resources as resource_id's
        available_resource_type_ids = set(available_items['resource_type_id'].tolist())

        # Check if all the ingredients for the current recipe are available
        if required_resource_type_ids.issubset(available_resource_type_ids):
            used_items = []

            # For each required resource, get one available item of the resource and remove it from available_items
            for resource_type_id in required_resource_type_ids:
                item_index = available_items[available_items['resource_type_id'] == resource_type_id].index[0]
                used_item = available_items.loc[item_index]
                used_items.append(used_item.resource_id) # assuming 'id' is the unique identifier of the resource item
                available_items.drop(item_index, inplace=True)

            # Record the crafted item and the used items (ResourceItems)
            crafted_items.append({
                'recipe_id': recipe.recipe_id,
                'item_quality': random.randint(5, 10),
                'resources': used_items
            })

    return crafted_items, available_items

# choose which subset of possible recipes to craft
def plan_crafting(inventory_items):
    available_items = inventory_items.copy()
    craftable_recipes = get_craftable_recipes(available_items)

    crafted_items = []

    if len(craftable_recipes.index) > 0:
      # Split the craftable_recipes DataFrame into potions and others
      potion_recipes = craftable_recipes[craftable_recipes.recipe_category == 'Potion']
      other_recipes = craftable_recipes[craftable_recipes.recipe_category != 'Potion']

      # Try to craft potions first
      if len(potion_recipes.index) > 0:
        crafted_items, available_items = craft_items(potion_recipes, available_items)

      # Then try to craft other items
      if len(other_recipes.index) > 0:
        other_crafted_items, available_items = craft_items(other_recipes, available_items)
        crafted_items.extend(other_crafted_items)

    return pd.DataFrame(crafted_items)

In [ ]:
from sqlalchemy.engine.row import Row
## Crafting function
def simulate_crafting(player, inventory_id, time):
    # get player info
    player_id = player['player_id']
    crafting_table_id = get_crafting_table_id(player_id)

    # figure out what to craft from resources in inventory
    inventory_items = get_resources_in_inventory(inventory_id)
    items_to_craft = plan_crafting(inventory_items)

    # craft each item
    for _ , item in items_to_craft.iterrows():
        # transfer resources from inventory to crafting table
        for resource_id in item['resources']:
            transfer_resource(resource_id, inventory_id, crafting_table_id, time)

        # create new item in crafting table collection
        recipe_id = item['recipe_id']
        item_id = create_item(recipe_id, item['item_quality'], crafting_table_id)

        # transfer item from crafting table to player
        item_transfer_id = transfer_item(item_id, crafting_table_id, inventory_id, time)

        create_crafting_event(random.randint(50, 100), recipe_id, item_transfer_id, crafting_table_id, time)

## Interaction Code

In [ ]:
def get_allowed_interaction_locations():
  locations_query = """
  SELECT location_id, location_group_name, locations.location_group_id, location_group_island_faction as location_faction
  FROM locations
  JOIN location_groups on locations.location_group_id = location_groups.location_group_id
  WHERE location_group_type = 'collection_area'
  """
  return(query_db(locations_query))

def get_player_location(player, allowed_locations, time):
    factions = ['Light', 'Shadow', 'Growth', 'Stability']

    # Determine the preferred island based on the time-shifted factions array
    preferred_island = player['player_favorite_island']
    preferred_faction = player['player_favorite_faction']

    # Get a dataframe of all possible locations
    possible_locations_df = allowed_locations

    # Create a list of factions that aligns with the locations in the dataframe
    location_factions = possible_locations_df['location_faction'].tolist()

    # Create a list of weights for the random.choices function
    weights = [.4 if faction == preferred_faction or faction == preferred_island else .1 for faction in location_factions]

    # Select a location based on the weights
    selected_location = random.choices(possible_locations_df['location_id'].tolist(), weights=weights, k=1)[0]

    # print(f"""player with preferred island {preferred_island} and faction {preferred_faction} going to {selected_location}""")

    return selected_location


def get_creature_location(creature, allowed_locations, time):
    factions = ['Light', 'Shadow', 'Growth', 'Stability']

     # Get the index of the creature's faction
    faction_index = factions.index(creature['creature_faction'])

    # Create a time-shifted version of the factions array
    time_shifted_factions = factions[time % len(factions):] + factions[:time % len(factions)]

    # Determine the preferred island based on the time-shifted factions array
    preferred_island = time_shifted_factions[faction_index]

    # Get a dataframe of all possible locations
    possible_locations_df = allowed_locations

    # Create a list of factions that aligns with the locations in the dataframe
    location_factions = possible_locations_df['location_faction'].tolist()

    # Create a list of weights for the random.choices function
    weights = [.4 if faction == creature['creature_faction'] or faction == preferred_island else .1 for faction in location_factions]

    # Select a location based on the weights
    selected_location_id = random.choices(possible_locations_df['location_id'].tolist(), weights=weights, k=1)[0]

    location_series = possible_locations_df[possible_locations_df['location_id'] == selected_location_id].iloc[0]
    return location_series


def decide_to_give_gift(player, creature):
    # if creature_faction is player_favorite_faction
    # or if creature_mood is lower than 50
    # be more likely to give gifts (return true)
    decision = random.choice([True, False])
    return decision

def select_random_gift(inventory_id):
    items = get_items_in_inventory(inventory_id)
    item_list = items['item_id'].tolist()

    if not item_list:  # The list is empty, so the inventory is empty
        return None
    else:
        gift = random.choice(item_list)
        return gift

def create_creature_sighting(time, creature_id, location_group_id):
    creature_sightings = Table('creature_sightings', metadata, autoload_with=engine)

    with engine.begin() as conn:
        ins = creature_sightings.insert().values(creature_sighting_time=time, creature_id=creature_id, location_group_id=location_group_id)
        result = conn.execute(ins)
    return result.inserted_primary_key[0]

def simulate_interactions(player, inventory_id, nearby_creatures, time):
    # Loop through the nearby creatures
    for _, creature in nearby_creatures.iterrows():
        # randomly select interactions, based on player.player_favorite_faction, creature_faction
        interaction = random.choice([True, False]) #select_random_interaction(player, creature)

        if (interaction):
            # interaction should return {player_id, creature_id, creature_mood, time}
            # decide whether to give a gift from available inventory items
            gift_given = decide_to_give_gift(player, creature)

            if gift_given:
                # Select a gift from the player's inventory
                gift_id = select_random_gift(inventory_id)

                if(gift_id):
                    # Create an item_transfer
                    item_transfer_id = transfer_item(gift_id, inventory_id, creature['collection_id'], time)

                    # Create a gifting_event with that item_transfer_id
                    gifting_event_id = create_gifting_event(player, creature, item_transfer_id, time)

                    # update_player_creature_relationship_data(player['player_id'], creature['creature_id'], 2)
                    create_interaction_event(player=player, creature=creature, location_id=player['location'], gifting_event_id=gifting_event_id, time=time)
                    boost_relationship_level(player['player_id'], creature['creature_id'], 3)
            else:
                create_interaction_event(player=player, creature=creature, location_id=player['location'], gifting_event_id=None, time=time)
                boost_relationship_level(player['player_id'], creature['creature_id'], 1)

    return

## Update code

In [ ]:
def player_creature_relationship_update(player_id, creature_id, change):
    print("update player creature relationship")
    return

def update_creature_stats():
    creatures_query = f"SELECT * FROM creatures;"
    creatures_df = query_db(creatures_query)
    opposite_factions = {'Light': 'Shadow', 'Shadow': 'Light', 'Growth': 'Stability', 'Stability': 'Growth'}

    recipe_ingredients = get_recipe_ingredients()

    for _ , creature in creatures_df.iterrows():
        inventory_id = creature['collection_id']
        items = get_items_in_inventory(inventory_id)

        effects = {
            'mood_effect': 0,
            'health_effect': 0
        }
        for _ , item in items.iterrows():
            recipe_id = item['recipe_id']
            ingredients_used = recipe_ingredients[recipe_ingredients['recipe_id'] == recipe_id]

            effects['mood_effect'] += item['recipe_base_mood_effect']
            effects['health_effect'] += item['recipe_base_health_effect']

            for _ , ingredient in ingredients_used.iterrows():
                base_mood_effect = ingredient['resource_type_base_mood_effect']
                base_health_effect = ingredient['resource_type_base_health_effect']

                if creature['creature_faction'] == ingredient['resource_type_faction']:
                    effects['mood_effect'] += 2 * base_mood_effect
                    effects['health_effect'] += 2 * base_health_effect

                elif creature['creature_faction'] == opposite_factions[ingredient['resource_type_faction']]:
                    effects['mood_effect'] -= base_mood_effect
                    effects['health_effect'] -= base_health_effect

        new_mood = creature['creature_mood'] + effects['mood_effect']
        new_health = creature['creature_health'] + effects['health_effect']
        update_creature_data(creature['creature_id'], creature_mood=new_mood, creature_health=new_health)
    return

# 🎮 Run Gameplay Simulation

## reset


In [ ]:
def clear_tables(table_name):
    disable_fk_check = "SET FOREIGN_KEY_CHECKS = 0"
    enable_fk_check = "SET FOREIGN_KEY_CHECKS = 1"
    delete_query = f"DELETE FROM {table_name}"

    with engine.begin() as conn:
        conn.execute(text(disable_fk_check))

    with engine.begin() as conn:
        conn.execute(text(delete_query))

    with engine.begin() as conn:
        conn.execute(text(enable_fk_check))

for table_name in ['interaction_events', 'gifting_events', 'crafting_events', 'resource_transfers', 'item_transfers', 'resources', 'items', 'creature_sightings']:
    clear_tables(table_name)

In [ ]:
def reset_simulation():
    # Create connection to the database
    with engine.begin() as conn:
        # Reset player_creature_relationships
        reset_relationships_query = """
        UPDATE player_creature_relationships
        SET player_creature_relationship_level = 5;
        """
        conn.execute(text(reset_relationships_query))

        # Reset creature_mood and creature_health for all creatures
        reset_creatures_query = """
        UPDATE creatures
        SET creature_mood = 50,
            creature_health = 50;
        """
        conn.execute(text(reset_creatures_query))

        # Alternatively, to set a particular faction to start with a lower mood
        #reset_creatures_query_faction = """
        #UPDATE creatures
        #SET creature_mood = CASE
        #    WHEN creature_faction = 'your_specific_faction' THEN 40
        #    ELSE 50
        #    END,
        #creature_health = 50;
        #"""
        #conn.execute(text(reset_creatures_query_faction))

        print("Simulation values have been reset.")

reset_simulation()

Simulation values have been reset.


## run

In [ ]:
from sqlalchemy.types import DOUBLE_PRECISION
import random

# Define the number of game days to simulate
days = 10

# Get all players
players_query = f"SELECT * FROM players;"
players_df = query_db(players_query)

# Loop over each day
for day in range(days):
    # Increment the game_time
    time = day
    allowed_locations = get_allowed_interaction_locations()

    # get all creatures
    creatures_query = f"SELECT * FROM creatures;"
    creatures_df = query_db(creatures_query)

    creatures_df['location_id'] = None
    for index , creature in creatures_df.iterrows():
        creature_id = creature['creature_id']
        new_location = get_creature_location(creature, allowed_locations, time)
        creatures_df.at[index, 'location_id'] = new_location['location_id']

        location_group_id = new_location['location_group_id']
        create_creature_sighting(time, creature_id, location_group_id)

    # create creature_locations distribution
    for index, player in players_df.iterrows():
        inventory_id = get_inventory_id(player['player_id'])

        simulate_foraging(player, inventory_id, time) # Get 5-10 items probabilistically based on favorite_island and favorite_species
        simulate_crafting(player, inventory_id, time) # Given items in player inventory, craft any craftable potions, then craft gifts or foods

        player['location'] = get_player_location(player, allowed_locations, time) # Decide probabilistically which location the players will visit
        nearby_creatures = creatures_df[creatures_df['location_id'] == player['location']] # Find which creatures are also there (to interact with)
        simulate_interactions(player, inventory_id, nearby_creatures, time) # Loop over possible creature interactions, decide whether to interact and/or gift from inventory

    # at end of day update creature mood and health based on gifted items
    update_creature_stats();
    print(f"day {day} complete")

day 0 complete
day 1 complete
day 2 complete
day 3 complete
day 4 complete
day 5 complete
day 6 complete
day 7 complete
day 8 complete
day 9 complete


# Todo

1. [DONE] Create Timestamp/Date-time

Each time_segment:

1. [DONE] Players Forage: Spawn rate by rarity, player alliance, favorite island. (When creating items and populating the ItemCollections table, Create the Item then Transfer it to the player's inventory. This will create a record of where that item came from.)

2. [DONE] Simulate Crafting (player):
  2a. get Player's Inventory ID and Crafting Table ID.
  2b. Identify craftable recipes from Player Inventory
  2c. Create TransferEvents for resources from Inventory to Player's Crafting Table.
  2d. Create new RecipeItem in Player's CraftingTable Inventory.
  2e. Create TransferEvent for RecipeItem from CraftingTableID to Player Inventory.

3. [DONE] Simulate Interacting and Gifting (player)
  (identify creature preference, preferred friend, probabilistically create an interaction event. Create Gifting Events). Update relationships and creature mood.

4. Update Creature stats based on mood and sickness logic, and items in the Creature's "inventory".

# random resource drop to player inventories

In [ ]:
import random

def forage_items(num_items: int):
    # Let's assume we have a list of possible resource_ids and recipe_ids
    resource_ids = query_db("SELECT id from Resources")['id'].tolist()  # replace with actual ids
    collection_ids = query_db("SELECT id from Collections WHERE type='inventory'")['id'].tolist()

    items = []
    for _ in range(num_items):
        item = {'resource_id': random.choice(resource_ids), 'collection_id': random.choice(collection_ids)}
        items.append(item)

    with engine.begin() as conn:
        for _ in range(num_items):
            r = random.choice(resource_ids)
            c = random.choice(collection_ids)
            q = 50
            conn.execute(
                ResourceItems.insert().values(quality=q, resource_id=r, collection_id=c)
            )

In [ ]:
forage_items(40)

ProgrammingError: ignored

In [ ]:
import random

def spawn_items(num_items: int):
    # Let's assume we have a list of possible resource_ids and recipe_ids
    resource_ids = query_db("SELECT id from Resources")['id'].tolist()  # replace with actual ids
    collection_ids = query_db("SELECT id from Collections WHERE type='player'")['id'].tolist()

    items = []
    for _ in range(num_items):
        item = {'resource_id': random.choice(resource_ids), 'collection_id': random.choice(collection_ids)}
        items.append(item)

    with engine.begin() as conn:
        for _ in range(num_items):
            r = random.choice(resource_ids)
            c = random.choice(collection_ids)
            q = 50
            conn.execute(
                ResourceItems.insert().values(quality=q, resource_id=r, collection_id=c)
            )

spawn_items(40)

## Random Additions

In [ ]:
get_items_in_inventory(111)

In [ ]:
query_db(f"""

SELECT resource_type, resource_quality FROM resources
JOIN collections on resources.collection_id = collections.collection_id
JOIN resource_types on resources.resource_type_id = resource_types.resource_type_id
WHERE resources.collection_id = 111

""")

,resource_type,resource_quality
0,Star Crystal,4
1,Star Crystal,8
2,Stormfruit,8


In [ ]:
def get_creature_inventory_id(creature_id):
    # Create the SQL query
    query = f"""
    SELECT collection_id FROM collections
    JOIN creatures ON collections.collection_name LIKE creatures.creature_name
    WHERE collection_type = 'creature'
    AND creature_id = {creature_id}
    """

    # Execute the query and return the result
    with engine.connect() as conn:
        result = pd.read_sql_query(text(query), conn)

    # Assuming there's only one collection per player of type 'player',
    # the following line will return the collection ID as an integer
    return result

     #['collection_id'].values[0]

get_creature_inventory_id(1)

,collection_id
